In [ ]:
import cv2
import numpy as np
import pandas as pd
df = pd.read_csv('annotations.tsv', sep=',')

def calc_iou(box1, box2):
    xl = max(box1['x'], box2['x'])
    yt = max(box1['y'], box2['y'])
    xr = min(box1['x'] + box1['width'], box2['x'] + box2['width'])
    yb = min(box1['y'] + box1['height'], box2['y'] + box2['height'])
    if xr < xl or yb < yt:
        return 0.0
    area = (xr - xl) * (yb - yt)
    box1_area = box1['width'] * box1['height']
    box2_area = box2['width'] * box2['height']
    union_area = box1_area + box2_area - area
    iou = area / union_area
    return iou

def detect_barcode(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    grad_x = np.absolute(grad_x)
    grad_x = np.uint8(255 * grad_x / np.max(grad_x))
    _, thresh = cv2.threshold(grad_x, 80, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 5))
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    closed = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel_open)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None
    sorted_contours = sorted(contours, key=cv2.contourArea, reverse=True)
    for contour in sorted_contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = cv2.contourArea(contour)
        if area < 1000:
            continue
        aspect_ratio = float(w) / h
        if aspect_ratio < 1.1:
            continue
        return {'x': x, 'y': y, 'width': w, 'height': h}
    return None

results = []

for idx, row in df.iterrows():
    image_path = row['filename']
    ground_truth = {
        'x': row['x_from'],
        'y': row['y_from'],
        'width': row['width'],
        'height': row['height']
    }
    detected = detect_barcode(image_path)
    if detected is None:
        iou = 0
        print(f"{image_path}: Штрих-код не найден")
    else:
        iou = calc_iou(ground_truth, detected)
        results.append(iou)
        print(f"{image_path}: IoU = {iou:.3f}")
        if iou < 0.8:
            print(f"  IoU меньше 0.8")

if results:
    avg_iou = np.mean(results)
    min_iou = np.min(results)
    max_iou = np.max(results)
    good_results = sum(1 for iou in results if iou >= 0.7)
    print(f"\nСтатистика по всем изображениям:")
    print(f"Средний IoU: {avg_iou:.3f}")
    print(f"Минимальный IoU: {min_iou:.3f}")
    print(f"Максимальный IoU: {max_iou:.3f}")
    print(f"Количество изображений: {len(results)}")
    print(f"Хороших результатов (IoU >= 0.8): {good_results}/{len(results)}")

In [ ]:
images/0b56af7e-386c-410a-8f46-74350f755d77--ru.4c7208d1-cba4-4539-967d-1c87ad0f6d2e.jpg: IoU = 0.875
images/6e042372-20ec-4a26-8340-3653cf0c681d--ru.407738ff-e321-4f6c-8a9d-d9c146d3d801.jpg: IoU = 0.830
images/788fb7f3-e813-4754-874f-3cc6eb19113d--ru.265581a1-33c3-486e-b9e8-0fc5d4818e42.jpg: IoU = 0.987
images/d05d50c0-0c19-4ca4-88ff-6f7058318a7f--ru.fb097934-ebc8-4b94-be1c-e3720092b0d6.jpg: IoU = 0.685
  IoU меньше 0.8
images/93cda132-8b66-476a-a112-8acf47658fee--ru.c9cabec9-8436-4b5c-9d1d-a00905371a3d.jpg: IoU = 0.884
images/a66f361d-1e84-4f65-992a-4636e3b4ef9a--ru.c1328ca2-08a4-4c51-9251-4ae0a172b9d0.jpg: IoU = 0.947
images/af464982-dc36-4ef1-89b8-9f899c26bd98--ru.05dda538-814b-423f-a33a-75c000046f47.jpg: IoU = 0.714
  IoU меньше 0.8
images/35019e8a-6a4c-463d-8a42-2b0631133bdc--ru.982ad218-0498-4416-9cf2-7f09ca97d0c2.jpg: IoU = 0.965
images/f1f950b5-be18-4335-8dc6-6bea7445062b--ru.1502c7bf-6de8-42f1-9f47-90987bad4911.jpg: IoU = 0.592
  IoU меньше 0.8
images/6f8eeeda-3403-4c81-b37d-f9bbfada0e65--ru.21344efc-b08b-40ea-8a1c-ee975c5ef4a4.jpg: IoU = 0.910
images/fe2886d7-955c-4909-a61e-6e388a379b71--ru.ed56e791-b8db-4589-94e6-1e7c43bd60b7.jpg: IoU = 0.908
images/5a71f3c4-ab19-4a78-939b-d4c7522d96ad--ru.8c807fc5-6dcc-4681-a855-0623d19b7a00.jpg: IoU = 0.933
images/67c49161-584c-4355-b885-4c4f748234d2--ru.de000ead-d47f-4969-b040-4fdc357be09d.jpg: IoU = 0.839
images/77252d88-2d4f-4741-a507-1aee82b4ca53--ru.9ebb3de8-8c88-4e43-9157-0cce279f91f2.jpg: IoU = 0.482
  IoU меньше 0.8
images/468376b5-89e0-4658-9471-c03f2cdf9cfc--ru.5cf83e98-8a44-48c6-b9a6-0c61409fdc1a.jpg: IoU = 0.307
  IoU меньше 0.8
images/18aac3cf-1bdc-42b4-af6a-a02bdf1423cd--ru.01d6afae-b6d2-4352-8b8b-7b499044c551.jpg: IoU = 0.883
images/52d310ea-3bcc-47a0-9cb5-4b294054a7be--ru.5678e1a4-1367-44cb-b2d2-1bae7fbadbff.jpg: IoU = 0.832
images/9185f86c-3aee-43f6-a758-febdbd9281c5--ru.86a9b5f5-fff1-4b3b-bdb2-5c6f1153ed7b.jpg: IoU = 0.848
images/09c4aef7-de37-45d9-8ccf-3be4a4409953--ru.f8b7d94c-b600-4f02-bf94-410861132882.jpg: IoU = 0.878
images/5423eb16-945b-433b-9668-ff6b1727b8aa--ru.a73d075e-544f-4b2f-8218-cf28f3d59dd5.jpg: IoU = 0.885
images/49e72b24-d106-496f-82e4-e3c3441e89e0--ru.b53ccdc1-a710-4412-8e53-8cef3c095468.jpg: IoU = 0.955
images/83e63f27-b540-433e-8969-da31616deea0--ru.8adf5d09-c98c-4617-aae2-0bf975ae8ad1.jpg: IoU = 0.889
images/08fb775c-1553-4dbe-9d57-3739e58278d8--ru.16e5f570-5ef0-4354-ac80-51f4de9734fe.jpg: IoU = 0.952
images/dd690770-73e0-41e0-bc74-87fc937c8e5f--ru.790971c2-81f3-44b1-867f-dbcfb5e156c5.jpg: IoU = 0.925
images/4a8d968f-68b5-41b5-986e-7011af1170d1--ru.acb65758-2995-4c01-a07b-6fab6063dc2b.jpg: IoU = 0.900
images/2d3601d0-b3f1-4580-b1e3-899a0df7d796--ru.79bfe362-dc95-49a2-b0ef-472b76b27d32.jpg: IoU = 0.946
images/a4f6df3b-8cb4-429c-87ef-65de4666e7f1--ru.8276720b-c579-4b0f-bc43-53b379c54efc.jpg: IoU = 0.972
images/3d1c5c67-f786-4864-b331-bde3fe5f58b8--ru.ffdd8b7a-32f9-4b9f-ab7b-a5fa2b5e1914.jpg: IoU = 0.810
images/758e22fa-7bc0-4d31-8c2b-19dd8173d2dc--ru.463e87bd-8bdf-4369-8afa-475a1c2d6459.jpg: IoU = 0.794
  IoU меньше 0.8
images/3a45d9ed-6369-42da-96f4-01a2462295bc--ru.b59e36dc-46b7-441c-9f4a-82ac06b73842.jpg: IoU = 0.848
images/58494e6d-fc82-4a32-aa33-0c14fa0562d3--ru.6ed014f0-811a-4686-b710-bce52791e777.jpg: IoU = 0.944
images/ff8c167e-0909-48a3-9a17-419d75109699--ru.1e9c4d27-04a7-4869-913b-e659f86949c6.jpg: IoU = 0.860
images/69502f61-1ca5-4305-b29c-5ff5422c9a61--ru.25d7828e-4022-43ed-82ef-a86f8dd74260.jpg: IoU = 0.900
images/c9ffc301-76cc-4713-8c37-ca3c90bb89c5--ru.fb940454-6152-46e7-ac6f-8d1986698b2b.jpg: IoU = 0.000
  IoU меньше 0.8
images/0fa79872-6f0b-4eeb-bddc-3a2291a96a4b--ru.c1ba6b5d-e583-4612-8ca5-f97eb88050a6.jpg: IoU = 0.000
  IoU меньше 0.8
images/2c7eaef3-a462-40e8-9270-7edaa32f4f67--ru.c67368cf-e479-411a-b457-11f7ae7493cd.jpg: IoU = 0.056
  IoU меньше 0.8
images/37a5d5c3-b5a0-457f-954d-1fcff5b977de--ru.b9ca1b28-2ccd-458d-b0de-5cdb7ae977ef.jpg: IoU = 0.797
  IoU меньше 0.8
images/83f43e88-df4d-4ce5-a901-1dbbe0e2aba1--ru.b05fe251-b9f2-4edb-8325-adb543059974.jpg: IoU = 0.000
  IoU меньше 0.8
images/4e8d7767-2e12-4d9a-ac60-84b94f1458e3--ru.a01ac30d-27e6-425e-b162-14773d6cbc3c.jpg: IoU = 0.752
  IoU меньше 0.8
images/76a1b4ac-9b47-4531-bf91-8ba02ad0859c--ru.9c290067-02c7-45ca-a88a-1656672e0337.jpg: IoU = 0.872
images/59205bd8-6f43-419b-a88a-6d33e6adfc8f--ru.bc5b8e19-e6f3-4b89-ac12-ecef49232fc8.jpg: IoU = 0.894
images/7b14d1b4-6035-44fe-8204-00b557b19791--ru.2f579d2c-cd88-4e33-ae86-1b1883eb7a81.jpg: IoU = 0.957
images/4a4af57d-8d1e-4701-81f2-de6334812317--ru.ade76553-7528-4c8b-84c2-dc72717c7490.jpg: IoU = 0.930
images/1c92ab0b-ab48-47b3-bdcd-fcd82b52ea19--ru.5a3b4133-a3c9-401b-b913-616495ea7984.jpg: IoU = 0.889
images/f155dcc8-153c-4ccb-afda-cc8ba5d4b2dd--ru.c2d60678-c9ab-4b12-8dbe-1e8022ca418c.jpg: IoU = 0.000
  IoU меньше 0.8
images/7a5ef483-4556-42be-9a22-db4df0cfda99--ru.c9907faf-afe9-4a35-b971-6b5702b7b9ab.jpg: IoU = 0.965
images/e67af742-54ad-4575-9a44-5a579201d818--ru.a8603ad4-6e47-4fc4-a32e-691e5fe2f9b4.jpg: IoU = 0.880
images/57d59edc-2c2f-4b5b-a792-9c6ffbfb403d--ru.21d1c62f-5444-4f24-8900-87cb95b2b055.jpg: IoU = 0.000
  IoU меньше 0.8
images/37add18b-a51b-4daa-9567-21063b08c53c--ru.73f16523-0c7c-4db7-9f49-1d0d16bcbcc4.jpg: IoU = 0.935
images/b3426db2-3799-419b-8745-27bc7b961dee--ru.e7476cad-7bf1-4b89-9c74-7591056c9004.jpg: IoU = 0.874
images/e2128466-ecc8-4e42-8d5b-482c2a5ca42c--ru.a7a30e1d-89b8-4a0a-817d-ba7e57b3b311.jpg: IoU = 0.839
images/d3335271-b494-4ff5-9dbf-d58ffdfd24cf--ru.93cd1d2e-060e-45e5-833c-47be4963d7ae.jpg: Штрих-код не найден
images/d80c553f-8eea-4c75-8870-65cf26c71469--ru.d64c655e-5b1d-448a-bbe9-9aa4deea7a15.jpg: IoU = 0.902
images/1c37e1fa-b1c8-4fea-9ba4-916c0d1580ba--ru.cb8daef9-6c7e-49df-8f44-33d2f44c1f2c.jpg: IoU = 0.945
images/818f55e6-0cf3-4df7-a3bb-b9d063d8e153--ru.101805e9-f3e4-42dc-80e6-3b49c4b9e911.jpg: IoU = 0.761
  IoU меньше 0.8
images/346f96cd-fe53-4dcd-b28c-55b58b5392b9--ru.043b8cea-061f-4dd3-9bcb-4113600e320c.jpg: IoU = 0.673
  IoU меньше 0.8
images/38fd8888-76a7-4c35-a3e8-a9426555b4de--ru.6137a32c-1bb2-44c1-8dcf-49371c1dad64.jpg: IoU = 0.824
images/3a83373a-f943-48c0-b331-a9ed5fae3f2a--ru.e5c6c02d-e507-4714-b0aa-3449c25aed0c.jpg: IoU = 0.856
images/96592587-bc94-45bf-8630-e861443af782--ru.4b6e0452-4b9b-41b1-925e-dd6344b6793c.jpg: IoU = 0.745
  IoU меньше 0.8
images/f20f263b-3a99-451b-8f60-90a440a23b60--ru.26ef1722-bdc5-44ae-adfa-2a541c805eb2.jpg: IoU = 0.539
  IoU меньше 0.8
images/b79b3a1c-3403-4a1a-9e0c-b33d8e2a2866--ru.45a7f970-dbb1-4a77-90cf-2f64e95386ae.jpg: IoU = 0.739
  IoU меньше 0.8
images/eb9b68b0-4ea5-42c3-95db-d24d90e5a622--ru.579adf62-88c6-4a7d-9ed9-28dd2e5e3fc5.jpg: IoU = 0.892
images/bd9089d0-127b-4b35-a543-ac2035ce8c1d--ru.6e68aeed-508d-468f-b485-fe214919d678.jpg: IoU = 0.917
images/3811eb7e-bb36-40ab-b42b-616ede97dbc1--ru.949d75d9-898e-4b5d-a498-622ec0145c83.jpg: IoU = 0.975
images/aeaf9f35-5865-463c-8ed0-6c01fcd5d4bc--ru.bfbcca81-b8d3-4686-8927-5ab7f2f8ba77.jpg: IoU = 0.903
images/3e36e1ba-414b-4e79-93b7-ad043693b44f--ru.c07726d3-5083-4cc3-a4db-b51ccff817a3.jpg: IoU = 0.876
images/91e6b74a-d92c-4e14-a11a-77d184a9e0a7--ru.a89de72e-e13c-4ae0-9160-7753ea2d269a.jpg: IoU = 0.932
images/79c0c2ed-3369-412a-8133-ebade5fdf428--ru.5b650ec3-d493-4365-98ab-1740becd3087.jpg: IoU = 0.944
images/bb56ecc6-1e4b-4a50-8088-9bf519e0a00b--ru.1c807f15-4cfb-4c51-9173-3b9a2653e28a.jpg: IoU = 0.787
  IoU меньше 0.8
images/94bd03b6-3ea4-4b42-a3bc-bcbb6487fa8d--ru.72cbdcbb-7881-43da-b50e-e8fadb1d9e14.jpg: IoU = 0.900
images/b02d4381-605c-48a4-b942-044fc7e44c65--ru.f9be9835-9b96-4c46-9473-910a41922bf1.jpg: IoU = 0.873
images/cce85bf0-1dab-4e3f-8837-ba09bd25a20d--ru.a1e3dbf1-a83c-47f8-9295-d8a99b084389.jpg: IoU = 0.908
images/b5c905af-95d7-4a15-abac-3ec01e01ff51--ru.b5d3fb19-8731-4a17-b326-edadfbabaf22.jpg: IoU = 0.837
images/4296efb5-f449-45bd-9d4a-2e63aed1940e--ru.0ddb9259-762e-4658-91b7-275cc1e5859d.jpg: IoU = 0.947
images/a751736c-6631-4ece-b5ba-e887799810a5--ru.78fd5a0e-7960-40db-8c82-57d016be84ad.jpg: IoU = 0.828
images/056793c5-dd56-4bf2-a2c4-7447290c51ee--ru.77d6ba32-ad28-42a2-ade9-267de19a696f.jpg: IoU = 0.958

Статистика по всем изображениям:
Средний IoU: 0.786
Минимальный IoU: 0.000
Максимальный IoU: 0.987
Количество изображений: 75
Хороших результатов (IoU >= 0.8): 63/75